In [2]:
import requests
import json
import os

In [3]:
folder = "test_endpoints"
os.makedirs(folder, exist_ok=True)

In [4]:
def _infer_schema(value): 
    if value is None: 
        return {"type": "null"} 
    if isinstance(value, bool): 
        return {"type": "boolean"} 
    if isinstance(value, int): 
        return {"type": "integer"} 
    if isinstance(value, float): 
        return {"type": "number"} 
    if isinstance(value, str): 
        return {"type": "string"} 
    if isinstance(value, list): 
        schema = {
            "type": "array"
        } 
        if value: 
            item_schemas = [_infer_schema(item) for item in value] 
            if all(item_schema == item_schemas[0] for item_schema in item_schemas):
                schema["items"] = item_schemas[0]
            else:
                schema["items"] = item_schemas
        return schema 
    if isinstance(value, dict): 
        properties = {} 
        for key, child_value in value.items(): 
            properties[key] = _infer_schema(child_value) 
        return { "type": "object", "properties": properties } 
    return { "type": "unknown" }

def explore_endpoint(
    url_v,
    endpoint,
    params=None,
    timeout=30
):
    BASE_URL = {
        "1": "https://statsapi.mlb.com/api/v1/",
        "1.1": "https://statsapi.mlb.com/api/v1.1/"
    }
    if endpoint.startswith("http://") or endpoint.startswith("https://"):
        url = endpoint
    else:
        endpoint = endpoint.lstrip("/")
        url = f"{BASE_URL[url_v]}/{endpoint}"

    response = requests.get(
        url,
        params=params,
        timeout=timeout
    )

    response.raise_for_status()

    data = response.json()

    return {
        "schema": _infer_schema(data),
        "response": data
    }

## Sports

In [5]:
sports = explore_endpoint(
    url_v="1", 
    endpoint="sports"
)

with open(f"{folder}/sports_schema.json", "w") as f:
    json.dump(sports['schema'], f, indent=4)
with open(f"{folder}/sports_response.json", "w") as f:
    json.dump(sports['response'], f, indent=4)

In [6]:
# MLB: id = 1 
MLB_SPORT_ID = 1

mlb = explore_endpoint(
    url_v="1", 
    endpoint=f"sports/{MLB_SPORT_ID}"
)

with open(f"{folder}/mlb_schema.json", "w") as f:
    json.dump(mlb['schema'], f, indent=4)
with open(f"{folder}/mlb_response.json", "w") as f:
    json.dump(mlb['response'], f, indent=4)

## Leagues

In [7]:
league = explore_endpoint(
    url_v="1",
    endpoint="league"
)

with open(f"{folder}/league_schema.json", "w") as f:
    json.dump(league['schema'], f, indent=4)
with open(f"{folder}/league_response.json", "w") as f:
    json.dump(league['response'], f, indent=4)

In [8]:
# AL: id = 103
AL_LEAGUE_ID = 103

al = explore_endpoint(
    url_v="1",
    endpoint=f"league/{AL_LEAGUE_ID}"
)

with open(f"{folder}/al_schema.json", "w") as f:
    json.dump(al['schema'], f, indent=4)
with open(f"{folder}/al_response.json", "w") as f:
    json.dump(al['response'], f, indent=4)

In [9]:
# NL: id = 104
NL_LEAGUE_ID = 104

nl = explore_endpoint(
    url_v="1",
    endpoint=f"league/{NL_LEAGUE_ID}"
)

with open(f"{folder}/nl_schema.json", "w") as f:
    json.dump(nl['schema'], f, indent=4)
with open(f"{folder}/nl_response.json", "w") as f:
    json.dump(nl['response'], f, indent=4)

## Divisions

In [10]:
al_divisions = explore_endpoint(
    url_v="1",
    endpoint="divisions",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", AL_LEAGUE_ID)
    ]
)

with open(f"{folder}/al_divisions_schema.json", "w") as f:
    json.dump(al_divisions['schema'], f, indent=4)
with open(f"{folder}/al_divisions_response.json", "w") as f:
    json.dump(al_divisions['response'], f, indent=4)

In [11]:
nl_divisions = explore_endpoint(
    url_v="1",
    endpoint="divisions",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", NL_LEAGUE_ID)
    ]
)

with open(f"{folder}/nl_divisions_schema.json", "w") as f:
    json.dump(nl_divisions['schema'], f, indent=4)
with open(f"{folder}/nl_divisions_response.json", "w") as f:
    json.dump(nl_divisions['response'], f, indent=4)

## Teams

In [12]:
al_teams = explore_endpoint(
    url_v="1",
    endpoint="teams",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", AL_LEAGUE_ID)
    ]
)

with open(f"{folder}/al_teams_schema.json", "w") as f:
    json.dump(al_teams['schema'], f, indent=4)
with open(f"{folder}/al_teams_response.json", "w") as f:
    json.dump(al_teams['response'], f, indent=4)

In [13]:
al_teams_id = {team['name']: team['id'] for team in al_teams['response']['teams']}
display(al_teams_id)

{'Athletics': 133,
 'Seattle Mariners': 136,
 'Tampa Bay Rays': 139,
 'Los Angeles Angels': 108,
 'Texas Rangers': 140,
 'Toronto Blue Jays': 141,
 'Baltimore Orioles': 110,
 'Minnesota Twins': 142,
 'Boston Red Sox': 111,
 'Chicago White Sox': 145,
 'Cleveland Guardians': 114,
 'New York Yankees': 147,
 'Detroit Tigers': 116,
 'Houston Astros': 117,
 'Kansas City Royals': 118}

In [14]:
nl_teams = explore_endpoint(
    url_v="1",
    endpoint="teams",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", NL_LEAGUE_ID)
    ]
)

with open(f"{folder}/nl_teams_schema.json", "w") as f:
    json.dump(nl_teams['schema'], f, indent=4)
with open(f"{folder}/nl_teams_response.json", "w") as f:
    json.dump(nl_teams['response'], f, indent=4)

In [15]:
nl_teams_id = {team['name']: team['id'] for team in nl_teams['response']['teams']}
display(nl_teams_id)

{'Pittsburgh Pirates': 134,
 'San Diego Padres': 135,
 'San Francisco Giants': 137,
 'St. Louis Cardinals': 138,
 'Arizona Diamondbacks': 109,
 'Philadelphia Phillies': 143,
 'Chicago Cubs': 112,
 'Atlanta Braves': 144,
 'Cincinnati Reds': 113,
 'Miami Marlins': 146,
 'Colorado Rockies': 115,
 'Los Angeles Dodgers': 119,
 'Washington Nationals': 120,
 'New York Mets': 121,
 'Milwaukee Brewers': 158}

## Venue

In [16]:
padres_venue_id = 2680
padres_venue = explore_endpoint(
    url_v="1",
    endpoint=f"venues/{padres_venue_id}",
    params=[
        ("hydrate", "fieldInfo")
    ]
)

with open(f"{folder}/padres_venue_schema.json", "w") as f:
    json.dump(padres_venue['schema'], f, indent=4)
with open(f"{folder}/padres_venue_response.json", "w") as f:
    json.dump(padres_venue['response'], f, indent=4)

## Roster

In [17]:
padres_team_id = 135
padres_roster = explore_endpoint(
    url_v="1",
    endpoint=f"teams/{padres_team_id}/roster/fullRoster"
)

with open(f"{folder}/padres_roster_schema.json", "w") as f:
    json.dump(padres_roster['schema'], f, indent=4)
with open(f"{folder}/padres_roster_response.json", "w") as f:
    json.dump(padres_roster['response'], f, indent=4)

## Player

In [24]:
player_id = 592518
player = explore_endpoint(
    url_v="1",
    endpoint=f"people/{player_id}"
)

with open(f"{folder}/player_schema.json", "w") as f:
    json.dump(player['schema'], f, indent=4)
with open(f"{folder}/player_response.json", "w") as f:
    json.dump(player['response'], f, indent=4)

## Schedule

In [19]:
schedule = explore_endpoint(
    url_v="1",
    endpoint="schedule",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("date", "2026-08-11"),
        ("hydrate", "team(standings)")
    ]
)

with open(f"{folder}/schedule_schema.json", "w") as f:
    json.dump(schedule['schema'], f, indent=4)
with open(f"{folder}/schedule_response.json", "w") as f:
    json.dump(schedule['response'], f, indent=4)

## Game

In [20]:
GAME_PK = schedule['response']['dates'][0]['games'][0]['gamePk']

### Content

In [21]:
content = explore_endpoint(
    url_v="1",
    endpoint=f"game/{GAME_PK}/content"
)

with open(f"{folder}/content_schema.json", "w") as f:
    json.dump(content['schema'], f, indent=4)
with open(f"{folder}/content_response.json", "w") as f:
    json.dump(content['response'], f, indent=4)

### Gumbo

In [22]:
gumbo = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live",
    params=[
            ("hydrate", "credits,alignment,preState")
        ]
)

with open(f"{folder}/gumbo_schema.json", "w") as f:
    json.dump(gumbo['schema'], f, indent=4)
with open(f"{folder}/gumbo_response.json", "w") as f:
    json.dump(gumbo['response'], f, indent=4)